### Installation
see https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-Alpaca.ipynb

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
print("PyTorch version:", torch.__version__)
print("Torch Built With CUDA:", torch.backends.cuda.is_built())
print("Current CUDA Device:", torch.cuda.current_device())
print("CUDA Version (Torch):", torch.version.cuda)

CUDA available: True
GPU: NVIDIA H100 80GB HBM3
PyTorch version: 2.5.1
Torch Built With CUDA: True
Current CUDA Device: 0
CUDA Version (Torch): 12.1


In [2]:
import pandas as pd
import time
from tqdm import tqdm

### Unsloth

In [24]:
model_name="unsloth/Meta-Llama-3.1-8B"


In [5]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# "unsloth/Meta-Llama-3.1-8B"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

==((====))==  Unsloth 2025.4.1: Fast Llama patching. Transformers: 4.51.3.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 1. Max memory: 79.209 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1. CUDA: 9.0. CUDA Toolkit: 12.1. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.28.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [6]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""
inputs = tokenizer(
[
    alpaca_prompt.format(
        "Continue the fibonnaci sequence.", # instruction
        "1, 1, 2, 3, 5, 8", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
tokenizer.batch_decode(outputs)

['<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nContinue the fibonnaci sequence.\n\n### Input:\n1, 1, 2, 3, 5, 8\n\n### Response:\n13\n<|end_of_text|>']

In [7]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""
inputs = tokenizer(
[
    alpaca_prompt.format(
        "What is the age of the animals?", # instruction
        "Juvenile pigs (approximately 3 months old) were used. ", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
tokenizer.batch_decode(outputs)

['<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nWhat is the age of the animals?\n\n### Input:\nJuvenile pigs (approximately 3 months old) were used. \n\n### Response:\nThe age of the animals is approximately 3 months old.\n<|end_of_text|>']

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [8]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2025.4.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


<a name="Data"></a>
### Data Prep
We now use the Alpaca dataset from [yahma](https://huggingface.co/datasets/yahma/alpaca-cleaned), which is a filtered version of 52K of the original [Alpaca dataset](https://crfm.stanford.edu/2023/03/13/alpaca.html). You can replace this code section with your own data prep.

**[NOTE]** To train only on completions (ignoring the user's input) read TRL's docs [here](https://huggingface.co/docs/trl/sft_trainer#train-on-completions-only).

**[NOTE]** Remember to add the **EOS_TOKEN** to the tokenized output!! Otherwise you'll get infinite generations!

If you want to use the `llama-3` template for ShareGPT datasets, try our conversational [notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Alpaca.ipynb)

For text completions like novel writing, try this [notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Mistral_(7B)-Text_Completion.ipynb).

In [9]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }
pass

from datasets import load_dataset
dataset = load_dataset("yahma/alpaca-cleaned", split = "train")
dataset = dataset.map(formatting_prompts_func, batched = True,)

In [10]:
dataset

Dataset({
    features: ['output', 'input', 'instruction', 'text'],
    num_rows: 51760
})

In [11]:
print(dataset[2])


{'output': "An atom is the basic building block of all matter and is made up of three types of particles: protons, neutrons, and electrons. The structure of an atom can be described as a nucleus at the center surrounded by a cloud of electrons.\n\nThe nucleus of an atom is made up of protons and neutrons. Protons are positively charged particles and neutrons are neutral particles with no charge. Both of these particles are located in the nucleus of the atom, which is at the center of the atom and contains most of the atom's mass.\n\nSurrounding the nucleus of the atom is a cloud of electrons. Electrons are negatively charged particles that are in constant motion around the nucleus. The electron cloud is divided into shells or orbitals, and each shell can hold a certain number of electrons. The number of electrons in the outermost shell, called the valence shell, determines the chemical properties of the atom. \n\nIn a neutral atom, the number of protons in the nucleus is equal to the n

<a name="Train"></a>
### Train the model
Now let's use Huggingface TRL's `SFTTrainer`! More docs here: [TRL SFT docs](https://huggingface.co/docs/trl/sft_trainer). We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support TRL's `DPOTrainer`!

In [12]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

In [13]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA H100 80GB HBM3. Max memory = 79.209 GB.
7.777 GB of memory reserved.


In [14]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 51,760 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040/8,000,000,000 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,1.577400
2,2.105900
3,1.676900
4,1.918200
5,1.751100
6,1.511500
7,1.133500
8,1.322300
9,1.192500
10,1.138100


In [15]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

71.663 seconds used for training.
1.19 minutes used for training.
Peak reserved memory = 8.068 GB.
Peak reserved memory for training = 0.291 GB.
Peak reserved memory % of max memory = 10.186 %.
Peak reserved memory for training % of max memory = 0.367 %.


In [25]:
model_name

'unsloth/Meta-Llama-3.1-8B'

In [26]:
model_name_str = model_name.lower().replace("-","_").replace("/","_")
model_name_str

'unsloth_meta_llama_3.1_8b'

In [27]:
# `model` is your LoRA‐wrapped model; `tokenizer` is the tokenizer you used.
model.save_pretrained(f"lora_model_{model_name_str}")      # saves only the adapter weights (and config) into lora_model/
tokenizer.save_pretrained(f"lora_model_{model_name_str}")  # saves tokenizer files side-by-side

('lora_model_unsloth_meta_llama_3.1_8b/tokenizer_config.json',
 'lora_model_unsloth_meta_llama_3.1_8b/special_tokens_map.json',
 'lora_model_unsloth_meta_llama_3.1_8b/tokenizer.json')

<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!

**[NEW] Try 2x faster inference in a free Colab for Llama-3.1 8b Instruct [here](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Unsloth_Studio.ipynb)**

In [16]:
# alpaca_prompt = Copied from above
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer(
[
    alpaca_prompt.format(
        "Continue the fibonnaci sequence.", # instruction
        "1, 1, 2, 3, 5, 8", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
tokenizer.batch_decode(outputs)

['<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nContinue the fibonnaci sequence.\n\n### Input:\n1, 1, 2, 3, 5, 8\n\n### Response:\n13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2584, 4181, 6765, 10946, 17711, 28657, 46368, 75025']

In [17]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer(
[
    alpaca_prompt.format(
        "What is the age of the used animals in this experiments? Return `AGE: <standardized age or descriptor>`", # instruction
        "Mice used in these studies were 2-3 month old males with a mixed C57BL/6 6 129S background", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
tokenizer.batch_decode(outputs)

['<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nWhat is the age of the used animals in this experiments? Return `AGE: <standardized age or descriptor>`\n\n### Input:\nMice used in these studies were 2-3 month old males with a mixed C57BL/6 6 129S background\n\n### Response:\nAGE: Young adult (2-3 months old)<|end_of_text|>']

In [18]:
instructions_simpler = """### TASK ###

EXTRACT THE AGE OF ANIMALS MENTIONED IN THE TEXT.  
RETURN ONLY THE AGE IN A STANDARDIZED FORMAT.  
IF NO AGE IS GIVEN, RETURN `"AGE NOT SPECIFIED"`.

---

### HOW TO THINK (CHAIN OF THOUGHTS) ###

1. READ the sentence carefully.
2. FIND any age-related phrases (e.g., "8 weeks", "P30", "adult", "3 months").
3. LINK each age phrase to the animal(s) it describes.
4. STANDARDIZE the format:  
   - Use `<number> <unit>` (e.g., `8 weeks`, `3 months`)  
   - Keep terms like `adult`, `juvenile`, `neonatal` unchanged  
5. IF no age is mentioned, write: `"AGE NOT SPECIFIED"`

---

### OUTPUT FORMAT ###

- `AGE: <standardized age or descriptor>`

---

### EXAMPLES ###

#### INPUT 1 ####  
Gene deletion was induced in male and female 12- to 20-week-old mice.  
#### OUTPUT 1 ####  
AGE: 12-20 weeks

#### INPUT 2 ####  
Six adult male WAG/Rij rats were used.  
#### OUTPUT 2 ####  
AGE: adult

#### INPUT 3 ####  
Juvenile pigs (approximately 3 months old) were used.  
#### OUTPUT 3 ####  
AGE: 3 months

#### INPUT 4 ####  
For Experiment 2, male young (3-months-old) and aged (23-months-old) rats were used.  
#### OUTPUT 4 ####  
AGE: 3 months, 23 months

#### INPUT 5 ####  
Twenty Sprague Dawley rats were used; no details were provided on age.  
#### OUTPUT 4 ####  
AGE: AGE NOT SPECIFIED

---

### WHAT NOT TO DO ###

- DO NOT include the whole sentence in the output  
- DO NOT include weight, sex, or strain  
- DO NOT guess the age  
- DO NOT omit the unit (e.g., weeks/months)  
- DO NOT ignore terms like "adult", "neonatal", or "juvenile"  
- DO NOT return multiple values or unformatted strings

---
ONLY OUTPUT THE AGE USING THE FORMAT ABOVE. NOTHING ELSE.
"""

In [19]:
binary_instruct_to_filter_sent = """YOU ARE A SCIENTIFIC TEXT CLASSIFIER. YOUR TASK IS TO DECIDE IF A GIVEN TEXT MENTIONS **THE AGE OF ANIMALS** USED IN AN EXPERIMENT.

###TASK###
READ THE TEXT AND ANSWER:
**DOES THE TEXT CONTAIN ANY INFORMATION ABOUT THE AGE OF THE ANIMALS (e.g., IN WEEKS, MONTHS, OR YEARS)?**

- RETURN "YES" IF THE TEXT SAYS HOW OLD THE ANIMALS ARE (e.g., "8–10 weeks old", "3 months of age").
- RETURN "NO" IF THERE IS NO AGE MENTIONED.

###CHAIN OF THOUGHT###
1. UNDERSTAND: READ THE TEXT CAREFULLY.
2. BASICS: LOOK FOR AGE TERMS LIKE "WEEKS", "MONTHS", "YEARS", OR "AGE".
3. BREAK DOWN: IS THERE A PHRASE THAT TELLS YOU HOW OLD THE ANIMALS ARE?
4. ANALYZE: IF YES, SAY "YES". IF NOT, SAY "NO".
5. FINAL ANSWER: OUTPUT "YES" OR "NO" ONLY.

###FEW-SHOT EXAMPLES###

####Example 1: (Positive)
TEXT:  
"CB1 and CB2 knockout mice (8–10 weeks of age) were used."  
ANSWER: YES

####Example 2: (Negative)
TEXT:  
"Blood samples were collected after behavioral tests."  
ANSWER: NO

####Example 3: (Positive)
TEXT:  
"Rats aged 3 months were housed in groups of four."  
ANSWER: YES

####Example 4: (Negative)
TEXT:  
"Mice were kept under standard laboratory conditions."  
ANSWER: NO

###WHAT NOT TO DO###
- DO NOT SAY "YES" IF THERE IS NO AGE MENTIONED
- DO NOT CONFUSE DURATIONS (e.g., “kept for 2 weeks”) WITH AGE
- DO NOT INCLUDE EXPLANATIONS — ONLY RETURN "YES" OR "NO"
"""

In [20]:
prompt_full_sentences = '''YOU ARE A TEXT CLASSIFIER THAT DECIDES IF A SENTENCE TELLS THE **AGE OF AN ANIMAL AT THE START OF AN EXPERIMENT**, OR IF IT TALKS ABOUT **SOME OTHER TIME INFORMATION** (WHICH SHOULD BE IGNORED).

###YOUR TASK###

- IF THE SENTENCE TELLS **THE AGE OF THE ANIMAL WHEN THE EXPERIMENT STARTED**, OUTPUT THAT AGE.
- IF THE SENTENCE TALKS ABOUT **TRAINING TIME, STUDY LENGTH, TIME AFTER INJURY**, OR OTHER TIME EVENTS, OUTPUT: `NOT AGE`.

###HOW TO THINK (CHAIN OF THOUGHT)###

1. READ the sentence carefully.
2. FIND any **time or age information** (e.g., “3 months,” “P21,” “postnatal day 5–7,” “8–10 weeks”).
3. ASK: Is this telling us how old the animals were when used?
   - YES → It is AGE.
   - NO → It is NOT AGE.
4. CHECK:
   - Words like “aged,” “adult (2–4 mos),” or “mice aged 8 weeks” usually mean AGE.
   - Words like “trained for 5 days,” “tested at 3 timepoints,” or “after 1 month” are NOT AGE.
5. GIVE THE FINAL ANSWER in this format:

```
AGE: <age or age range with unit>  — if age at experiment start
AGE: NOT AGE     — if it's any other time info
```

###EXAMPLES###

INPUT:
The rats were trained two times a day for 5 days .
OUTPUT:
AGE: NOT AGE

INPUT:
Adult ( 2 - 4 mos ) male C57BL/6 mice were used in these studies .
OUTPUT:
AGE: 2-4 months

INPUT:
At the time of the experiment, the young adult animals were 4 months of age, and the aged animals were 18 months of age.
OUTPUT:
AGE: 4 months, 18 months

INPUT:
Recordings were taken at 1, 3, and 5 months post-injury .
OUTPUT:
AGE: NOT AGE

INPUT:
Eight-week-old female C57BL/6J mice were housed in groups of five.
OUTPUT:
AGE: 8 weeks

INPUT:
Male adult Wistar rats, weighing between 190–230 g, were procured from the Experimental Animal Center.
OUTPUT:
AGE: adult 

###WHAT NOT TO DO###

- DO NOT extract how long training or testing lasted.
- DO NOT confuse “post-injury,” “postnatal days after birth,” or “study length” with animal age.
- NEVER guess the age if it's not clearly mentioned.
- NEVER output the whole sentence — only the AGE or "NOT AGE".
'''

In [21]:
inputs = tokenizer(
[
    alpaca_prompt.format(
        instructions_simpler,        
        "Mice used in these studies were 2-3 month old males with a mixed C57BL/6 6 129S background", # instruction
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 500, use_cache = True)
tokenizer.batch_decode(outputs)

['<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\n### TASK ###\n\nEXTRACT THE AGE OF ANIMALS MENTIONED IN THE TEXT.  \nRETURN ONLY THE AGE IN A STANDARDIZED FORMAT.  \nIF NO AGE IS GIVEN, RETURN `"AGE NOT SPECIFIED"`.\n\n---\n\n### HOW TO THINK (CHAIN OF THOUGHTS) ###\n\n1. READ the sentence carefully.\n2. FIND any age-related phrases (e.g., "8 weeks", "P30", "adult", "3 months").\n3. LINK each age phrase to the animal(s) it describes.\n4. STANDARDIZE the format:  \n   - Use `<number> <unit>` (e.g., `8 weeks`, `3 months`)  \n   - Keep terms like `adult`, `juvenile`, `neonatal` unchanged  \n5. IF no age is mentioned, write: `"AGE NOT SPECIFIED"`\n\n---\n\n### OUTPUT FORMAT ###\n\n- `AGE: <standardized age or descriptor>`\n\n---\n\n### EXAMPLES ###\n\n#### INPUT 1 ####  \nGene deletion was induced in male and female 12- to 20-week-old mice.  

In [20]:
input_sentence = """One month old male 3xTg - AD mice ( n = 9 ) were treated with 10 mM L - Carnosine ( Sigma - Aldrich ) in standard tap water for a period of 11 - 13 months .."""

In [30]:
negative_sentence = "Blood serum was collected 1 week after the Step-down test (122 days after the first injection), by orbital cavity bleeding after anaesthetization."  


In [31]:
inputs = tokenizer(
[
    alpaca_prompt.format(
        binary_instruct_to_filter_sent,        
        negative_sentence, # instruction
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 500, use_cache = True)
tokenizer.batch_decode(outputs)

['<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nYOU ARE A SCIENTIFIC TEXT CLASSIFIER. YOUR TASK IS TO DECIDE IF A GIVEN TEXT MENTIONS **THE AGE OF ANIMALS** USED IN AN EXPERIMENT.\n\n###TASK###\nREAD THE TEXT AND ANSWER:\n**DOES THE TEXT CONTAIN ANY INFORMATION ABOUT THE AGE OF THE ANIMALS (e.g., IN WEEKS, MONTHS, OR YEARS)?**\n\n- RETURN "YES" IF THE TEXT SAYS HOW OLD THE ANIMALS ARE (e.g., "8–10 weeks old", "3 months of age").\n- RETURN "NO" IF THERE IS NO AGE MENTIONED.\n\n###CHAIN OF THOUGHT###\n1. UNDERSTAND: READ THE TEXT CAREFULLY.\n2. BASICS: LOOK FOR AGE TERMS LIKE "WEEKS", "MONTHS", "YEARS", OR "AGE".\n3. BREAK DOWN: IS THERE A PHRASE THAT TELLS YOU HOW OLD THE ANIMALS ARE?\n4. ANALYZE: IF YES, SAY "YES". IF NOT, SAY "NO".\n5. FINAL ANSWER: OUTPUT "YES" OR "NO" ONLY.\n\n###FEW-SHOT EXAMPLES###\n\n####Example 1: (Positive)\nTEXT:

In [22]:
input_full_section = """
Materials and Methods

Animals. All animal studies were performed in male New Zealand White rabbits weighing 2.4–3.0 kg and approximately 10–13 weeks of age (obtained from Charles River Canada, Senneville, QC, Canada). All procedures were conducted in accordance with the Guide for the Care and Use of Laboratory Animals as adopted and promulgated by the U.S. National Institutes of Health, and approved by Merck Research Laboratories' Animal Care and Use Committee.

Rabbit Model of Cerebral Microembolic Signals Derived from FeCl$_{3}$-Induced Carotid Arterial Thrombosis. The rabbit model of cerebral MES was induced by FeCl$_{3}$ injury of the carotid artery as described in detail elsewhere (Zhou et al., 2016a,b). Briefly, animals were anesthetized with a cocktail (intramuscular, 50 mg/kg ketamine HCl [Pfizer, New York, NY], and 5 mg/kg xylazine [Lloyd Laboratories, Shenandoah, IA]) as a nonrecovery procedure. The left common carotid artery was surgically exposed. A Doppler flow probe (Model 1.5 or 2.0 PRB; Transonic Systems, Ithaca, NY) connected to a flowmeter (Model T403; Transonic Systems) was used to measure the artery blood flow with continuous data acquisition by a PowerLab 16/35 and LabChart Pro system (AD Instruments, Colorado Springs, CO).

Thrombosis was induced by applying two pieces of filter papers (7.4 mm in diameter, 0.5 mm thick each, above and beneath the vessel; Life Technologies, Grand Island, NY) presaturated with 30% FeCl$_{3}$ (anhydrous 98%, cat. no. 169430050; ACROS Organics/Thermo Fisher Scientific, Waltham, MA) to the adventitial surface of the vessel. A piece of para-film was placed underneath the vessel to protect the surrounding tissue from injury. The filter papers were removed after 5 minutes followed by washout of residue FeCl$_{3}$ with warm saline.

The blood flow was monitored for 60 minutes from the time of FeCl$_{3}$ application (as time 0). Integrated carotid blood flow over 60 minutes was measured by area under the curve (AUC), calculated by the trapezoidal rule, and expressed as the percentage of control blood flow as described previously elsewhere (Wong et al., 2008b). At end of study (i.e., 60 minutes after FeCl$_{3}$ injury), the wet thrombus weight was measured using a balance with a detection limit of 0.001 mg (Mettler Toledo Excellence Plus XP Series Analytical Balances; Mettler-Toledo, Columbus, OH).

For MES detection, we used the clinical SONARA transcranial Doppler system (Nicolet Natus Neurology, Middleton, WI) and continuously monitored the blood flow velocity and MES in the ipsilateral middle cerebral artery (MCA) for 60 minutes upon FeCl$_{3}$ injury. A pulse wave 2-MHz probe (o.d. 11.3 mm, 90 mm long, focused at 12 to 25 mm, customized by MTB Medizintechnik Basler AG, Regensdorf, Switzerland) was fixed by a flexible-arm magnetic base holder (McMaster-Carr, Princeton, NJ) at the posterior end of zygomatic bone of the rabbits, at an angle of ∼80 degrees against the buccal surface. The MCA was insonated at a depth between 19 to 22 mm, as described in detail elsewhere (Zhou et al., 2016a). MES (defined as high-intensity transient signals) was recorded and confirmed based on the criteria defined by the International Consensus Committee (CCotNICH Symposium, 1995; Ringelstein et al., 1998) as described elsewhere (Zhou et al., 2016a,b).

Rabbit Cuticle Bleeding Model. The rabbit cuticle bleeding time model was performed as described elsewhere (Wong et al., 2008a). Briefly, rabbits were anesthetized, and the apex of the cuticle was cut with a razor blade. The wound site was superfused with 37°C lactated Ringer’s solution to allow blood to flow freely. The bleeding time was recorded when bleeding ceased, with a maximal bleeding time set for 30 minutes. Three nail cuticles per rabbit were measured, and an average of the bleeding time from the three cuticles was considered as one data point.

Drug Administration. The FXIa inhibitor compound 1 (structure shown in Fig. 1), with a molecular weight of 596.6 Daltons, was synthesized after the procedures described in detail elsewhere (Corte et al., 2011) at Merck Research Laboratories (Kenilworth, NJ). Table 1 summarizes the potency and selectivity parameters of the compound (more details on our methods are found in the next section).

For the efficacy study (i.e., the 30% FeCl$_{3}$-induced carotid arterial thrombosis and cerebral MES study described earlier), various doses (0–3 mg/kg/h) of compound 1 or vehicle (35% hydroxypropyl β-cyclodextrin in 10 mM phosphate buffer, pH 7.0) were continuously infused at 2 ml/kg through the marginal ear vein. The i.v. dosing started 60 minutes before vessel injury and ended upon study completion. A dose of 5 mg/kg/h compound 1 was infused for the cuticle bleeding study; otherwise, the dosing regimen was the same as that which was used for the efficacy studies.

The dosing regimen for compound 1 was predicted to accomplish sufficient exposures in our study based on a previous internal standard pharmacokinetics study in rats (not shown), and it was validated for selected time points in the current rabbit MES and cuticle bleeding models under the current dosing regimen. Apixaban (a FXa inhibitor synthesized at Merck; Zhou et al., 2016b) at 3 mg/kg/h, i.v., was used as a control for the cuticle bleeding study.

In Vitro Protease Inhibition Assays. Human FXIa was purchased from Sekisui Diagnostics (Lexington, MA). Rabbit FIIa and FXa were purchased from Enzyme Research Laboratories. Rabbit plasma kallikrein, FIXa, and FVIIa were prepared internally (Merck Research Laboratories, Kenilworth, NJ). Rabbit FXIa was purified internally after activating rabbit FXI with human FXIIa, both purchased from Enzyme Research Laboratories (South Bend, IN). Rabbit FXIIa was prepared by Evotec (Princeton, NJ). The 7-amido-4-thifluoromethylcoumarin (AFC) containing fluorescence substrate CH3SO2-cyclohexyl-Gly-Gly-Arg-AFC (cyclohexyl-G-G-R-AFC) was custom synthesized by CPC Scientific (Sunnyvale, CA), and the substrates N-CBZ-Gly-Pro-Arg-AFC (Z-G-P-R-AFC), and n-acetyl-Gly-Pro-Arg-AFC (acetyl-K-P-R-AFC) were purchased from Sigma-Aldrich (St. Louis, MO).

All enzymatic reactions were performed in 50 mM HEPES, 150 mM NaCl, 5 mM CaCl$_{2}$, and 0.1% polyethylene glycol (8000 nominal molecular weight) at pH 7.4 and 25°C using Corning 3575 384-well assay plates (Corning, NY). The fluorescence signal was measured after 60 minutes of elapsed reaction time using an EnVision 2101 plate reader set at 405 nm excitation and 510 nm emission (PerkinElmer, Waltham, MA). The calculated percentage inhibition values were plotted versus the inhibitor concentration to determine IC$_{50}$. The IC$_{50}$ values were subsequently converted to K$_{I}$ using the Cheng-Prusoff equation (Zhang and Windsor, 2013).

Plasma Drug Exposure and Ex Vivo Clotting Time Assays. Blood samples were collected into sodium citrate (at 3.2% final concentration) Vacutainers (Becton Dickinson, Franklin Lakes, NJ) from either the central ear artery or the carotid artery at terminal bleed. The blood samples were centrifuged at 2000 g for 15 minutes at 4°C for plasma preparation and were evaluated for pharmacokinetic and ex vivo clotting time assays.

For compound 1 pharmacokinetic analysis, the plasma samples, plasma standards, and quality control stock (10 μl plasma) were assessed after protein precipitation in an acetonitrile crash solution (300 μl) containing labetalol (200 nM), diclofenac (200 nM), imipramine (200 nM), and alprazolam (100 nM). After automated sample preparation, the supernatant was then analyzed by liquid chromatography and tandem mass spectrometry for compound 1 using a Thermo Scientific LX-2 system (Thermo Fisher Scientific, Waltham, MA) for liquid chromatography and MSD Sciex API 5500 Q-Trap (Applied Biosystems, Forster City, CA) for the mass spectrometry analysis.

Specifically, water and 0.1% formic acid were used as the mobile phase A solution, and acetonitrile and 0.1% formic acid as the mobile phase B solution. An autosampler wash 1 containing 2% ammonium hydroxide in 5 mM aqueous ammonium acetate, acetone, isopropanol, acetonitrile, and methanol (1/1/1/1/1, v/v/v/v/v), and autosampler wash 2 containing methanol, water, and formic acid (10/89/1, v/v/v) were used. The MonChrom C18, 100 × 2.0 mm, 3 μm column was used at 50°C. A gradient solution containing 95%–5% of mobile phase A solution in combination with 5%–95% of mobile phase B solution was used to elute the column. Plasma drug exposure for apixaban was analyzed as described elsewhere (Zhou et al., 2016b).

Ex vivo plasma activated partial thromboplastin time (aPTT) and prothrombin time (PT) were determined by standard methods using aPTT-XL (Pacific Haemostasis, Waltham, MA) and TriniCLOT PT Excel (Tcoag, Bray, Ireland), respectively, on a KC4 Delta coagulation analyzer (Tcoag, Bray, Ireland).

Data Analysis and Statistics. Data are presented as mean ± standard error (S.E.) and were analyzed using one-way analysis of variance followed by Bonferroni post hoc test in GraphPad Prism (version 7; GraphPad Software, La Jolla, CA) for comparison among groups at different doses. Paired t test was applied for comparisons between paired groups. EC$_{50}$, defined as doses for half-maximal effect, was determined by a nonlinear four-parameter dose response curve fit using GraphPad Prism under the dose ranges described in each figure legend. A chi-square and Fisher’s test was applied to determine the association between drug exposure at each treatment dose and efficacy or pharmacodynamic readout. P < 0.05 was considered statistically significant.
"""

In [23]:
inputs = tokenizer(
[
    alpaca_prompt.format(
        """EXTRACT THE AGE OF ANIMALS MENTIONED IN THE TEXT.  
RETURN ONLY THE AGE IN A STANDARDIZED FORMAT.  
IF NO AGE IS GIVEN, RETURN `"AGE NOT SPECIFIED""",        
        input_full_section, # instruction
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 500, use_cache = True)
tokenizer.batch_decode(outputs)

Unsloth: Input IDs of length 2544 > the model's max sequence length of 2048.
We shall truncate it ourselves. It's imperative if you correct this issue first.


['<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nEXTRACT THE AGE OF ANIMALS MENTIONED IN THE TEXT.  \nRETURN ONLY THE AGE IN A STANDARDIZED FORMAT.  \nIF NO AGE IS GIVEN, RETURN `"AGE NOT SPECIFIED\n\n### Input:\n\nMaterials and Methods\n\nAnimals. All animal studies were performed in male New Zealand White rabbits weighing 2.4–3.0 kg and approximately 10–13 weeks of age (obtained from Charles River Canada, Senneville, QC, Canada). All procedures were conducted in accordance with the Guide for the Care and Use of Laboratory Animals as adopted and promulgated by the U.S. National Institutes of Health, and approved by Merck Research Laboratories\' Animal Care and Use Committee.\n\nRabbit Model of Cerebral Microembolic Signals Derived from FeCl$_{3}$-Induced Carotid Arterial Thrombosis. The rabbit model of cerebral MES was induced by FeCl$_{3

### run over full age dataset

In [18]:
df_age = pd.read_csv("./age_annotations.csv")


In [19]:
df_age.head()

,Unnamed: 0,doc_id,ent_type,ent_text,s_idx,e_idx,doc_id_unique
0,1,My_pdf69new_method,age,All animal studies were performed in male New ...,34,219,My_pdf69new_method_34_219
1,3,My_pdf764_method,age,Adult male SD rats (180-200 g) provided by the...,46,295,My_pdf764_method_46_295
2,5,My_pdf484new_method,age,"For all experiments described, larvae from 0 t...",1068,1157,My_pdf484new_method_1068_1157
3,8,My_pdf238new_method,age,Experiments were carried out in a double-blind...,28,219,My_pdf238new_method_28_219
4,9,My_pdf737_method,age,The experiments were performed on adult (10-12...,47,178,My_pdf737_method_47_178


In [21]:
df_sentences = pd.read_csv("./sent_with_age_kw_sent_classifier.csv")
df_sentences['doc_id_unique'] = df_sentences['doc_id'] + "_" + df_sentences['sentence_id'].astype(str).str.strip()

df_sentences = df_sentences[df_sentences['predicted_label']==1]

df_sentences.head()

,Unnamed: 0,doc_id,sentence_id,tokens,ner_tags,sent_txt,entity_types,age,has_age_kw,doc_id_unique,predicted_label,true_label
7,287,My_pdf746_method,7,"['Adult', '(', '2', '-', '4', 'mos', ')', 'mal...","['B-age', 'I-age', 'I-age', 'I-age', 'I-age', ...","Adult ( 2 - 4 mos ) male C57BL/6 , Lmx1b f / f...","['strain', 'age']",1,True,My_pdf746_method_7,1,1
16,547,My_pdf860_method,66,"['Male', 'mice', '(', '10', '-', '14', 'weeks'...","['B-age', 'I-age', 'I-age', 'I-age', 'I-age', ...","Male mice ( 10 - 14 weeks old , ICR ) received...","['strain', 'age']",1,True,My_pdf860_method_66,1,1
26,769,My_pdf174new_method,8,"['3,3', ""'"", '-N', '-', 'diaminobenzidine', 't...","['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', ...","3,3 ' -N - diaminobenzidine tetrahydrochloride...","['strain', 'age']",1,True,My_pdf174new_method_8,1,1
27,772,My_pdf174new_method,11,"['Four', '-', 'to', 'six', '-', 'week', '-', '...","['B-age', 'I-age', 'I-age', 'I-age', 'I-age', ...",Four - to six - week - old female BALB / c mic...,"['strain', 'age']",1,True,My_pdf174new_method_11,1,1
28,836,My_pdf174new_method,75,"['Sixty', '4', '-', 'to', '6', '-', 'week', '-...","['B-age|B-animals-number|B-randomization', 'I-...",Sixty 4 - to 6 - week - old female BALB / c mi...,"['strain', 'randomization', 'animals-number', ...",1,True,My_pdf174new_method_75,1,1


In [22]:
df_sentences.shape

(464, 12)

In [16]:
def extract_animal_age_unsloth(text, model, tokenizer, instructions):
    inputs = tokenizer(
        [
            alpaca_prompt.format(
                instructions,        
                text, # instruction
                "", # output - leave this blank for generation!
            )
        ], return_tensors = "pt").to("cuda")
        
    outputs = model.generate(**inputs, max_new_tokens = 500, use_cache = True)
    res = tokenizer.batch_decode(outputs)
    return res[0].split("Response:")[1].replace("<|end_of_text|>","").strip()
    

In [17]:
def run_predictions_and_collect_unsloth(df, text_col, model, tokenizer, instructions, output_path=None):
    predictions = []

    for i, row in tqdm(df.iterrows(), total=len(df)):
        doc_id = row["doc_id_unique"]
        text = row[text_col]

        try:
            age_prediction = extract_animal_age_unsloth(text, model, tokenizer, instructions)
        except Exception as e:
            age_prediction = f"ERROR: {str(e)}"

        prediction_row = {
            "doc_id_unique": doc_id,
            "ent_text": text,
            "age_prediction": age_prediction
        }

        predictions.append(prediction_row)

        if output_path:
            # Append row directly to file (write header only on first row)
            pd.DataFrame([prediction_row]).to_csv(
                output_path, mode='a', header=(i == 0), index=False
            )

    return pd.DataFrame(predictions)


In [29]:
model_name="unsloth/Meta-Llama-3.1-8B"
save_name = model_name.split("/")[1].lower()
start_time = time.time()

predictions = run_predictions_and_collect_unsloth(
    df_sentences,
    "sent_txt",
    model,
    tokenizer,
    prompt_full_sentences, #prompt_full_sentences, #instructions_simpler,
    output_path=f"predictions/classif_sent_from_methods/age_{save_name}_task_specific_prompt.csv"
)

end_time = time.time()
elapsed = end_time - start_time
print(f"\nCompleted predictions in {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")

100%|██████████| 464/464 [03:27<00:00,  2.24it/s]


Completed predictions in 207.19 seconds (3.45 minutes)


In [ ]:
save_name

 You can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [ ]:
# alpaca_prompt = Copied from above
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer(
[
    alpaca_prompt.format(
        "Continue the fibonnaci sequence.", # instruction
        "1, 1, 2, 3, 5, 8", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)

<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Continue the fibonnaci sequence.

### Input:
1, 1, 2, 3, 5, 8

### Response:
13, 21, 34, 55, 89, 144<|end_of_text|>


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("lora_model")  # Local saving
tokenizer.save_pretrained("lora_model")
# model.push_to_hub("your_name/lora_model", token = "...") # Online saving
# tokenizer.push_to_hub("your_name/lora_model", token = "...") # Online saving

('lora_model/tokenizer_config.json',
 'lora_model/special_tokens_map.json',
 'lora_model/tokenizer.json')

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model) # Enable native 2x faster inference

# alpaca_prompt = You MUST copy from above!

inputs = tokenizer(
[
    alpaca_prompt.format(
        "What is a famous tall tower in Paris?", # instruction
        "", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)

<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
What is a famous tall tower in Paris?

### Input:


### Response:
One of the most famous and iconic tall towers in Paris is the Eiffel Tower. Standing at 324 meters (1,063 feet) tall, this wrought iron tower is a symbol of the city and a must-see attraction for tourists from all over the world.<|end_of_text|>


You can also use Hugging Face's `AutoModelForPeftCausalLM`. Only use this if you do not have `unsloth` installed. It can be hopelessly slow, since `4bit` model downloading is not supported, and Unsloth's **inference is 2x faster**.

In [ ]:
if False:
    # I highly do NOT suggest - use Unsloth if possible
    from peft import AutoPeftModelForCausalLM
    from transformers import AutoTokenizer
    model = AutoPeftModelForCausalLM.from_pretrained(
        "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        load_in_4bit = load_in_4bit,
    )
    tokenizer = AutoTokenizer.from_pretrained("lora_model")

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [31]:
ls /scratch/sdonev

llms/  out_animals_nr_ner/


In [28]:
ls /shares/animalwelfare.crs.uzh

cadmus/  clinical_trials_ner/  goldhamster/  llms/  PsyNamic/  triomics/


In [30]:
# Merge to 16bit
if True: model.save_pretrained_merged("/scratch/sdonev/llms", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if True: model.save_pretrained_merged("/scratch/sdonev/llms", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if False: model.save_pretrained_merged("model", tokenizer, save_method = "lora",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "lora", token = "")

Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 270.54 out of 377.53 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


  0%|          | 0/32 [00:00<?, ?it/s]


RuntimeError: [enforce fail at inline_container.cc:603] . unexpected pos 576 vs 470

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("model", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "hf/model", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "",
    )

Now, use the `model-unsloth.gguf` file or `model-unsloth-Q4_K_M.gguf` file in llama.cpp or a UI based system like Jan or Open WebUI. You can install Jan [here](https://github.com/janhq/jan) and Open WebUI [here](https://github.com/open-webui/open-webui)

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>
